# TARDIS
This notebook loads the dataset, cleans the data, handles missing values, and creates new features for further analysis.

In [67]:
import pandas as pd
import numpy as np

## Load and Inspect the Dataset

In [68]:
df = pd.read_csv("dataset.csv")
print("(Nb rows, Nb columns): ", df.shape)
df.info()

(Nb rows, Nb columns):  (40, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   date               40 non-null     object 
 1   departure_station  39 non-null     object 
 2   arrival_station    39 non-null     object 
 3   scheduled_time     37 non-null     object 
 4   actual_time        36 non-null     object 
 5   delay_minutes      30 non-null     float64
dtypes: float64(1), object(5)
memory usage: 2.0+ KB


In [69]:
df.head()

,date,departure_station,arrival_station,scheduled_time,actual_time,delay_minutes
0,2024-02-01,Paris,Marseille,08:00,08:10,10.0
1,2024-02-01,Bordeaux,Lyon,09:15,09:20,5.0
2,2024-02-02,Lyon,Paris,07:30,07:40,10.0
3,2024-02-02,Marseille,Bordeaux,10:00,10:30,30.0
4,2024-02-03,Nice,Lille,12:45,12:50,5.0


In [70]:
print(df.columns)

Index(['date', 'departure_station', 'arrival_station', 'scheduled_time',
       'actual_time', 'delay_minutes'],
      dtype='object')


In [71]:
nb_duplicates = df.duplicated().sum()

if nb_duplicates > 0:
    df = df.drop_duplicates()

print("Nb duplicates: ", nb_duplicates)
print("(Nb rows, Nb columns): ", df.shape)

Nb duplicates:  0
(Nb rows, Nb columns):  (40, 6)


## Data Cleaning

In [72]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")

df["scheduled_time"] = pd.to_datetime(
    df["scheduled_time"].astype(str).str.replace("-", ":"),
    format="%H:%M",
    errors="coerce"
).dt.time

df["actual_time"] = pd.to_datetime(
    df["actual_time"].astype(str).str.replace("-", ":"),
    format="%H:%M",
    errors="coerce"
).dt.time

df["delay_minutes"] = pd.to_numeric(df["delay_minutes"], errors="coerce")
df.loc[df["delay_minutes"] < 0, "delay_minutes"] = np.nan

In [73]:
df["departure_station"] = df["departure_station"].fillna("Unknown").str.strip()
df["arrival_station"] = df["arrival_station"].fillna("Unknown").str.strip()

In [74]:
print(df.isnull().sum())

date                 26
departure_station     0
arrival_station       0
scheduled_time       16
actual_time          22
delay_minutes        11
dtype: int64


The dataset is checked for missing values so that incomplete rows or invalid values can be handled before feature creation.

In [75]:
df = df.dropna(subset=["date", "scheduled_time", "actual_time", "delay_minutes"])
print("(Nb rows, Nb columns): ", df.shape)

(Nb rows, Nb columns):  (10, 6)


Rows with missing values in the key columns are removed because these fields are necessary for reliable analysis and later feature engineering.

## Feature Engineering

New features are created from the original columns to make the dataset easier to analyze. In this section, temporal features such as year, month, quarter, season, and scheduled hour are added, as well as route, peak-hour, and delay-category features.

In [76]:
df['Year'] = df['date'].dt.year
df['Month'] = df['date'].dt.month

In [77]:
df["Month name"] = df["date"].dt.month_name()
df["Quarter"] = df["date"].dt.quarter

In [78]:
df["Season"] = df["Month"].map({
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Autumn", 10: "Autumn", 11: "Autumn"
})

In [79]:
df["Route"] = df["departure_station"] + " -> " + df["arrival_station"]

In [80]:
df["scheduled_hour"] = df["scheduled_time"].apply(
    lambda x: x.hour if pd.notnull(x) else np.nan
)

In [81]:
df["is_peak_hour"] = df["scheduled_hour"].apply(
    lambda x: 1 if x in [7, 8, 9, 17, 18, 19] else 0
)

In [82]:
df["delay_category"] = pd.cut(
    df["delay_minutes"],
    bins=[-1, 5, 15, 30, float("inf")],
    labels=["On time / slight", "Short delay", "Medium delay", "Long delay"]
)

In [83]:
df[["Year", "Month", "scheduled_hour", "is_peak_hour", "delay_category", "Route"]].head()

,Year,Month,scheduled_hour,is_peak_hour,delay_category,Route
0,2024,2,8,1,Short delay,Paris -> Marseille
1,2024,2,9,1,On time / slight,Bordeaux -> Lyon
2,2024,2,7,1,Short delay,Lyon -> Paris
3,2024,2,10,0,Medium delay,Marseille -> Bordeaux
4,2024,2,12,0,On time / slight,Nice -> Lille


In [84]:
df.to_csv('cleaned_dataset.csv', index=False)